In [1]:
import pandas as pd

# Carrega o arquivo CSV
df = pd.read_csv('results_moss_regressors.csv')

# 1. Quantidade total de datasets ÚNICOS
total_datasets = df['dataset'].nunique()
print(f"Total de datasets únicos: {total_datasets}")

# 2. Lista com os nomes de cada dataset
nomes_datasets = df['dataset'].unique()
print("Nomes dos datasets:", nomes_datasets)

# 3. Contagem de repetições/experimentos por dataset
contagem_por_dataset = df['dataset'].value_counts()
print(contagem_por_dataset)

Total de datasets únicos: 40
Nomes dos datasets: ['mhr' 'yeast' 'phishing' 'hcv' 'cmc'
 'dataset_1457_amazon-commerce-reviews.csv'
 'dataset_44478_amazon-commerce-reviews_seed_0_nrows_2000_nclasses_10_ncols_100_stratify_True.csv'
 'dataset_44479_amazon-commerce-reviews_seed_1_nrows_2000_nclasses_10_ncols_100_stratify_True.csv'
 'dataset_44480_amazon-commerce-reviews_seed_2_nrows_2000_nclasses_10_ncols_100_stratify_True.csv'
 'dataset_44481_amazon-commerce-reviews_seed_3_nrows_2000_nclasses_10_ncols_100_stratify_True.csv'
 'dataset_44482_amazon-commerce-reviews_seed_4_nrows_2000_nclasses_10_ncols_100_stratify_True.csv'
 'Mfeat.csv' 'obesity' 'image_seg' 'molecular' 'abalone'
 'academic-success' 'waveform-v1' 'page_block' 'digits' 'digits.csv'
 'satellite' 'wine-quality' 'isolet' 'fashion-mnist_test.csv'
 '!dataset_372_internet_usage.csv' 'hand_digits' 'nursery' 'dry-bean'
 'letter' 'letter.csv' 'Avila.csv' 'chess' 'Chessgame.csv' 'shuttle'
 'fashion-mnist_train.csv' 'connect-4' 'fashion

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import numpy as np
import os
import shutil

# ============================================================
# CONFIG
# ============================================================
CSV_PATH = "results_moss_regressors.csv"
COLUNA_CLASSES = "n_classes_original"

OS_DIR = "boxplots_por_classe"

# Limpa a pasta de saída para não misturar com PNGs/PDFs de execuções antigas
shutil.rmtree(OS_DIR, ignore_errors=True)
os.makedirs(OS_DIR, exist_ok=True)

# ============================================================
# PUBLICATION STYLE
# ============================================================
plt.rcParams.update({
    "font.family":        "serif",
    "font.serif":         ["Times New Roman", "Times", "DejaVu Serif"],
    "font.size":          10,
    "axes.labelsize":     10,
    "xtick.labelsize":    9,
    "ytick.labelsize":    9,
    "axes.linewidth":     0.4,
    "xtick.major.width":  0.0,
    "ytick.major.width":  0.0,
    "xtick.minor.width":  0.0,
    "ytick.minor.width":  0.0,
    "xtick.major.size":   0.0,
    "ytick.major.size":   0.0,
    "xtick.minor.size":   0.0,
    "ytick.minor.size":   0.0,
    "xtick.direction":    "in",
    "ytick.direction":    "in",
    "xtick.top":          False,
    "ytick.right":        False,
    "axes.grid":          True,
    "axes.grid.axis":     "y",
    "grid.linewidth":     0.4,
    "grid.alpha":         0.5,
    "grid.color":         "#aaaaaa",
    "figure.dpi":         300,
    "savefig.dpi":        300,
    "savefig.bbox":       "tight",
    "savefig.pad_inches": 0.05,
})

# ============================================================
# LOAD CSV
# ============================================================
df_master = pd.read_csv(CSV_PATH)

# ============================================================
# UNIFY MOSS VARIANTS (BASE, ISO, HYBRID + TODOS OS REGRESSORES
# DO CALIBRADOR)
# ============================================================

# Regras diretas (sem grupo de captura) para as variantes que já
# existiam antes do experimento de regressores.
df_master["modelo"] = df_master["modelo"].replace(
    {
        r"MoSS_BASE_\d+":           "MoSS_BASE",
        r"MoSS_ISO_CALIBRATED_\d+": "MoSS_ISO",
        r"MoSS_HYBRID_\d+":         "MoSS_HYBRID",
    },
    regex=True,
)

# Regra genérica com grupo de captura: qualquer
# "MoSS_<REGRESSOR>_CALIBRATED_<n_classes>" vira "MoSS_<REGRESSOR>".
# Cobre RF, ExtraTrees, GBR, KNN, Ridge, MLP, SVR — e qualquer outro
# nome que você adicionar em REGRESSOR_FACTORIES no script do
# experimento, sem precisar tocar neste plot de novo.
df_master["modelo"] = df_master["modelo"].str.replace(
    r"^MoSS_([A-Za-z0-9]+)_CALIBRATED_\d+$",
    r"MoSS_\1",
    regex=True,
)

# ============================================================
# RENAME MODELS
# ============================================================
df_master["modelo"] = df_master["modelo"].replace(
    {
        "EMQ_BCTS_QUAPY": None,
        "EMQ_QUAPY":      None,
        "EMQ_BCTS_MLQ":   "EMQ_BCTS",
        "EMQ_MLQ":        "EMQ",
    }
)

modelos_remover = ["EMQ_BCTS_QUAPY", "EMQ_QUAPY"]
df_master = df_master[~df_master["modelo"].isin(modelos_remover)]
df_master = df_master[df_master["modelo"].notna()]

# ============================================================
# LOOP
# ============================================================
max_classes_real = int(min(df_master[COLUNA_CLASSES].max(), 50))

for min_classes in range(3, max_classes_real + 1):

    df_filtrado = df_master[
        (df_master[COLUNA_CLASSES] >= min_classes) &
        (df_master[COLUNA_CLASSES] <= 50)
    ].copy()

    if df_filtrado.empty:
        print(f"Sem dados para o intervalo {min_classes} a 50. Parando o loop.")
        break

    dataset_mean = (
        df_filtrado.groupby(["dataset", "modelo"])["erro"]
        .mean()
        .reset_index()
    )

    dataset_mean["rank_dataset"] = (
        dataset_mean.groupby("dataset")["erro"]
        .rank(method="average")
    )

    ordem_rank = (
        dataset_mean.groupby("modelo")["rank_dataset"]
        .median()
        .sort_values()
        .index
        .tolist()
    )

    n_models = len(ordem_rank)

    # ============================================================
    # LABELS (BASE/ISO/HYBRID + TODOS OS REGRESSORES DO CALIBRADOR)
    # ============================================================
    label_map = {
        "MoSS_BASE":       r"$\mathrm{SMQ}$",
        "MoSS_ISO":        r"$\mathrm{SMQ}_\mathrm{ISO}$",
        "MoSS_HYBRID":     r"$\mathrm{SMQ}_\mathrm{HYB}$",
        "MoSS_RF":         r"$\mathrm{SMQ}_\mathrm{RF}$",
        "MoSS_ExtraTrees": r"$\mathrm{SMQ}_\mathrm{ET}$",
        "MoSS_GBR":        r"$\mathrm{SMQ}_\mathrm{GBR}$",
        "MoSS_KNN":        r"$\mathrm{SMQ}_\mathrm{KNN}$",
        "MoSS_Ridge":      r"$\mathrm{SMQ}_\mathrm{Ridge}$",
        "MoSS_MLP":        r"$\mathrm{SMQ}_\mathrm{MLP}$",
        "MoSS_SVR":        r"$\mathrm{SMQ}_\mathrm{SVR}$",
        "EMQ_BCTS":        r"$\mathrm{EMQ}_\mathrm{BCTS}$",
        "EMQ":             r"$\mathrm{EMQ}$",
        "KDEyML":          r"$\mathrm{KDEy}_\mathrm{ML}$",
        "KDEyCS":          r"$\mathrm{KDEy}_\mathrm{CS}$",
        "KDEyHD":          r"$\mathrm{KDEy}_\mathrm{HD}$",
    }

    display_labels = [label_map.get(m, m) for m in ordem_rank]

    palette = sns.color_palette("Spectral", n_colors=n_models)

    fig, ax = plt.subplots(figsize=(7.2, 3))

    sns.boxplot(
        data=dataset_mean,
        x="modelo",
        y="rank_dataset",
        hue="modelo",
        order=ordem_rank,
        hue_order=ordem_rank,
        palette=palette,
        dodge=False,
        legend=False,
        width=0.55,
        linewidth=0.8,
        flierprops=dict(
            marker="o",
            markerfacecolor="none",
            markeredgecolor="#555555",
            markeredgewidth=0.6,
            markersize=3.5,
        ),
        medianprops=dict(color="black", linewidth=1.4),
        whiskerprops=dict(linewidth=0.8),
        capprops=dict(linewidth=0.8),
        boxprops=dict(linewidth=0.8),
        ax=ax,
    )

    ax.set_xlabel("Methods", labelpad=6)
    ax.set_ylabel("Median Rank", labelpad=6)

    ax.set_ylim(0, 9)
    ax.yaxis.set_major_locator(ticker.MultipleLocator(2))
    ax.yaxis.set_minor_locator(ticker.NullLocator())

    for spine in ax.spines.values():
        spine.set_linewidth(0.4)
        spine.set_color("#aaaaaa")

    ax.tick_params(which="both", length=0)

    ax.set_xticks(range(n_models))
    ax.set_xticklabels(display_labels, rotation=40, ha="right", rotation_mode="anchor")

    plt.tight_layout()

    nome_arquivo = f"rank_boxplot_{min_classes}_a_50"
    #plt.savefig(f"{OS_DIR}/{nome_arquivo}.pdf")
    plt.savefig(f"{OS_DIR}/{nome_arquivo}.png", dpi=300)

    plt.close(fig)

    print(f"Salvo: {OS_DIR}/{nome_arquivo}.png (Iteração {min_classes} a 50)")

print("\nProcesso concluído! Verifique a pasta 'boxplots_por_classe'.")

Salvo: boxplots_por_classe/rank_boxplot_3_a_50.png (Iteração 3 a 50)
Salvo: boxplots_por_classe/rank_boxplot_4_a_50.png (Iteração 4 a 50)
Salvo: boxplots_por_classe/rank_boxplot_5_a_50.png (Iteração 5 a 50)
Salvo: boxplots_por_classe/rank_boxplot_6_a_50.png (Iteração 6 a 50)
Salvo: boxplots_por_classe/rank_boxplot_7_a_50.png (Iteração 7 a 50)
Salvo: boxplots_por_classe/rank_boxplot_8_a_50.png (Iteração 8 a 50)
Salvo: boxplots_por_classe/rank_boxplot_9_a_50.png (Iteração 9 a 50)
Salvo: boxplots_por_classe/rank_boxplot_10_a_50.png (Iteração 10 a 50)
Salvo: boxplots_por_classe/rank_boxplot_11_a_50.png (Iteração 11 a 50)
Salvo: boxplots_por_classe/rank_boxplot_12_a_50.png (Iteração 12 a 50)
Salvo: boxplots_por_classe/rank_boxplot_13_a_50.png (Iteração 13 a 50)
Salvo: boxplots_por_classe/rank_boxplot_14_a_50.png (Iteração 14 a 50)
Salvo: boxplots_por_classe/rank_boxplot_15_a_50.png (Iteração 15 a 50)
Salvo: boxplots_por_classe/rank_boxplot_16_a_50.png (Iteração 16 a 50)
Salvo: boxplots_por_

In [ ]:
verificar se sem rodar os modelos sem moss para ver quanto tempo dura - é muito rápido em um dia deve terminar todas as repetições
testar outras tecnicas de calibração multiclasse no smq iso tipo temperature scaling e dirthclet
smqrf testar outros regresores - deixar queto por enquanto 
hibrido vs outros metodos de oversampling
